# GPU Benchmark Notebook

This notebook tests GPU functionality and performance.

In [ ]:
# GPU Info
import torch
print("=== GPU Information ===")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}")
        print(f"  Memory: {props.total_memory / (1024**3):.1f} GB")
        print(f"  Compute capability: {props.major}.{props.minor}")

In [ ]:
# GPU Memory Stress Test
import torch
print("=== GPU Memory Test ===")

total_mem = torch.cuda.get_device_properties(0).total_memory
total_gb = total_mem / (1024**3)
print(f"Total GPU memory: {total_gb:.1f} GB")

tensors = []
allocated_gb = 0
chunk_gb = 2

try:
    while True:
        t = torch.zeros(1024, 1024, 1024, dtype=torch.float16, device="cuda")
        tensors.append(t)
        allocated_gb += chunk_gb
        if allocated_gb % 20 == 0:
            print(f"Allocated {allocated_gb} GB")
except RuntimeError:
    pass
finally:
    del tensors
    torch.cuda.empty_cache()

print(f"Max usable: {allocated_gb} GB / {total_gb:.1f} GB ({100*allocated_gb/total_gb:.0f}%)")

In [ ]:
# Matrix Multiply Benchmark
import torch
import time

print("=== Matrix Multiply Benchmark ===")
size = 8192
a = torch.randn(size, size, device="cuda", dtype=torch.float16)
b = torch.randn(size, size, device="cuda", dtype=torch.float16)

# Warmup
for _ in range(3):
    c = torch.matmul(a, b)
torch.cuda.synchronize()

# Benchmark
start = time.time()
iterations = 10
for _ in range(iterations):
    c = torch.matmul(a, b)
torch.cuda.synchronize()
elapsed = time.time() - start

flops = 2 * size**3 * iterations
tflops = flops / elapsed / 1e12
print(f"Matrix size: {size}x{size}")
print(f"Time: {elapsed:.3f}s for {iterations} iterations")
print(f"Performance: {tflops:.1f} TFLOPS")

In [ ]:
# Simple ML Training Test
import torch
import torch.nn as nn
import time

print("=== Simple ML Training Test ===")

model = nn.Sequential(
    nn.Linear(1024, 4096),
    nn.ReLU(),
    nn.Linear(4096, 4096),
    nn.ReLU(),
    nn.Linear(4096, 1024)
).cuda()

optimizer = torch.optim.Adam(model.parameters())
criterion = nn.MSELoss()

batch_size = 256
x = torch.randn(batch_size, 1024, device="cuda")
y = torch.randn(batch_size, 1024, device="cuda")

# Training loop
start = time.time()
for i in range(100):
    optimizer.zero_grad()
    output = model(x)
    loss = criterion(output, y)
    loss.backward()
    optimizer.step()
torch.cuda.synchronize()
elapsed = time.time() - start

print(f"100 training iterations: {elapsed:.2f}s")
print(f"Final loss: {loss.item():.4f}")
print("Training test PASSED")